# 04 Measure Clustering

Assignment step 7: group final measures into statistical domains while preserving unclustered/noise measures.


In [1]:
from pathlib import Path
import csv
import json

import pyarrow.parquet as pq

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent


def read_json(relative_path: str):
    path = ROOT / relative_path
    return json.loads(path.read_text(encoding="utf-8")) if path.exists() else {"missing": str(path)}


def csv_rows(relative_path: str, limit: int | None = None):
    csv.field_size_limit(2_147_483_647)
    path = ROOT / relative_path
    if not path.exists():
        return []
    with path.open("r", encoding="utf-8-sig", newline="") as file:
        rows = list(csv.DictReader(file))
    return rows if limit is None else rows[:limit]


def csv_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    if not path.exists():
        return None
    return len(csv_rows(relative_path))


def parquet_count(relative_path: str) -> int | None:
    path = ROOT / relative_path
    return pq.read_table(path).num_rows if path.exists() else None


## Clustering Summary


In [2]:
metrics = read_json("report/clustering_metrics.json")
{
    "measure_count": metrics.get("measure_count"),
    "cluster_count": metrics.get("cluster_count"),
    "coverage": metrics.get("coverage"),
    "unclustered_count": metrics.get("unclustered_count"),
    "baseline": metrics.get("baseline"),
    "validation": metrics.get("validation"),
    "manual_review": metrics.get("manual_review"),
}


{'measure_count': 632,
 'cluster_count': 35,
 'coverage': 0.5743670886075949,
 'unclustered_count': 269,
 'baseline': {'cluster_count': 42,
  'distance_threshold': 'configured',
  'method': 'agglomerative'},
 'validation': {'failures': [], 'passed': True},
 'manual_review': {'completed_count': 0,
  'review_sample': 'outputs\\clustering\\run_eb9d50c4e3e315dafd9d\\manual_cluster_review_sample.csv',
  'sample_count': 100,
  'status': 'pending'}}

## Domain Distribution


In [3]:
read_json("report/clustering_metrics.json").get("domain_distribution", {})


{'agriculture, forestry, and fisheries': 33,
 'cross-domain or other': 414,
 'economy and finance': 56,
 'education': 23,
 'industry, trade, and services': 16,
 'labour market': 40,
 'transport': 50}

## Cluster Examples


In [4]:
csv_rows("outputs/measure_clusters.csv", 8)


[{'term_id': 'term_20d6b4a1cc5307075745',
  'term': 'Accidents in waterway transport',
  'cluster_id': 'cluster_84a10ec4eb87',
  'domain': 'cross-domain or other',
  'membership_probability': '0.661999',
  'is_representative': 'false',
  'labeling_method': 'embedding_threshold_fallback',
  'evidence': '{"classification_run_id": "run_75b90575bb9e48ce7918", "domain_scores": {"environment and energy": 0.6404064612368662, "health": 0.7509626400393876, "population and demography": 0.7226856594867099, "social conditions and equality": 0.7042526606772864, "transport": 0.6728705276947698}, "source": "hdbscan_pearl_small"}'},
 {'term_id': 'term_2c16cafab319d30eed0d',
  'term': 'Accidents involving the transport of dangerous goods',
  'cluster_id': 'cluster_84a10ec4eb87',
  'domain': 'cross-domain or other',
  'membership_probability': '0.661999',
  'is_representative': 'false',
  'labeling_method': 'embedding_threshold_fallback',
  'evidence': '{"classification_run_id": "run_75b90575bb9e48ce791